# Classificação de risco de crédito com métodos clássicos de Machine Learning

**Dataset:** German Credit (Statlog), UCI Machine Learning Repository, licença CC BY 4.0 (Hofmann, 1994).
**Problema:** classificar cada solicitante como bom pagador (`0`) ou mau pagador (`1`).
**Projeto:** MVP da disciplina *Engenharia de Software para Sistemas Inteligentes*.

## Roteiro deste notebook

1. Preparação do ambiente
2. Carga dos dados
3. Análise exploratória
4. Seleção de atributos
5. Separação entre treino e teste (*holdout*)
6. Transformação dos dados (padronização e codificação)
7. Validação cruzada
8. Modelagem com configurações padrão
9. Otimização de hiperparâmetros
10. Avaliação no conjunto de teste
11. Comparação visual dos resultados
12. Escolha e exportação do modelo
13. Teste do modelo exportado
14. Principais achados e pontos de atenção
15. Conclusão

## Como executar

No Colab, use **Ambiente de execução → Executar tudo**. O notebook lê o CSV diretamente do GitHub, então é preciso ter conexão com a internet. A execução completa leva menos de dois minutos com a CPU padrão, sem necessidade de GPU.

## 1. Preparação do ambiente

Começamos importando as bibliotecas e fixando uma semente aleatória (`SEED = 42`). A semente faz com que a divisão dos dados, a validação cruzada e os modelos que usam aleatoriedade produzam sempre o mesmo resultado. Assim, quem executar o notebook verá os mesmos números.

Também registramos a versão do scikit-learn, porque o modelo exportado só pode ser carregado por uma versão compatível da biblioteca.

In [ ]:
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
SEED = 42
print("scikit-learn:", sklearn.__version__, "| pandas:", pd.__version__)

## 2. Carga dos dados

O dataset está em um repositório público no GitHub. Usamos a URL *raw*, que entrega o arquivo CSV puro, e o `pandas` lê o arquivo direto da internet. Assim o notebook não depende de upload manual e qualquer pessoa obtém exatamente a mesma base.

O arquivo tem 1000 linhas e 21 colunas: 20 atributos do solicitante e a coluna-alvo `credit_risk`.

In [ ]:
URL = "https://raw.githubusercontent.com/joaoclaudio25/german_credit/main/german_credit.csv"
df = pd.read_csv(URL)
print("Linhas e colunas:", df.shape)
df.head()

## 3. Análise exploratória

Antes de modelar, verificamos a qualidade dos dados e o equilíbrio das classes.

- **Valores ausentes:** se houver valores nulos, eles precisariam de tratamento antes do treino. O dataset não os possui, então nenhuma imputação é necessária.
- **Balanceamento da classe-alvo:** a proporção entre bons e maus pagadores é o principal fato a considerar na escolha das métricas. Se uma classe for muito maior que a outra, a acurácia sozinha pode enganar.

In [ ]:
print("Valores nulos no total:", int(df.isnull().sum().sum()))

contagem = df["credit_risk"].value_counts().rename({0: "bons (0)", 1: "maus (1)"})
print(contagem)
print((contagem / len(df)).round(3))

## 4. Seleção de atributos

Usamos 8 dos 20 atributos originais, escolhidos na etapa anterior do projeto:

- **Categóricos:** `status_checking_account`, `credit_history`, `purpose`, `savings_account` e `employment_since`.
- **Numéricos:** `duration_months`, `credit_amount` e `age`.

Deixamos de fora dois atributos sensíveis, `personal_status_sex` (sexo e estado civil) e `foreign_worker` (nacionalidade). Usá-los para decidir crédito pode gerar discriminação, e a decisão de excluí-los foi tomada com base nesse risco ético, e não por desempenho.

In [ ]:
FEATURES_CAT = ["status_checking_account", "credit_history", "purpose",
                "savings_account", "employment_since"]
FEATURES_NUM = ["duration_months", "credit_amount", "age"]
FEATURES = FEATURES_CAT + FEATURES_NUM

X = df[FEATURES]
y = df["credit_risk"]
print("Atributos usados:", X.shape[1], "| alvo:", y.name)

## 5. Separação entre treino e teste (*holdout*)

Separamos 20% dos dados como **conjunto de teste**. Esses dados ficam guardados até o fim do notebook e não são usados para escolher modelo nem hiperparâmetros. Eles servem para estimar o desempenho em casos novos.

O `stratify=y` mantém a mesma proporção de maus pagadores nos dois conjuntos. Como a classe de maus pagadores é minoritária, isso evita um conjunto de teste com proporção diferente da base real.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
print(f"Treino: {len(X_train)} linhas | Teste: {len(X_test)} linhas")
print("Proporção de maus no treino:", round(y_train.mean(), 3),
      "| no teste:", round(y_test.mean(), 3))

## 6. Transformação dos dados

Os modelos precisam de números, e os atributos estão em escalas e formatos diferentes:

- **Categóricos** (`A11`, `A32` etc.) são convertidos em colunas 0/1 com `OneHotEncoder`. Cada categoria vira uma coluna própria, sem criar uma falsa ordem entre os códigos.
- **Numéricos** são **padronizados** com `StandardScaler`: subtraímos a média e dividimos pelo desvio-padrão, de modo que cada atributo fique com média 0 e desvio 1. Isso importa para o KNN e o SVM, que calculam distâncias. Uma variável como `credit_amount`, que vai de centenas a milhares, dominaria uma variável como `age` se não fosse padronizada.

A árvore de decisão não depende de escala, mas aplicamos a mesma transformação a todos os modelos para que a comparação seja justa.

Colocamos essas transformações dentro de um **pipeline**, junto com o modelo. Assim, a média e o desvio-padrão são calculados apenas com os dados de treino de cada etapa de validação cruzada. Se a padronização fosse feita antes da separação, parte da informação do teste vazaria para o treino.

In [ ]:
def criar_preprocessamento():
    return ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), FEATURES_CAT),
        ("num", StandardScaler(), FEATURES_NUM),
    ])

def criar_pipeline(modelo):
    return Pipeline([("prep", criar_preprocessamento()), ("clf", modelo)])

# Demonstração: o que o pré-processamento produz a partir do treino
preparado = criar_preprocessamento().fit(X_train)
print("Atributos antes:", X_train.shape[1], "| depois da transformação:",
      preparado.transform(X_train).shape[1])

## 7. Validação cruzada

Com um único conjunto de treino, a escolha de um modelo ou de um hiperparâmetro poderia depender da sorte da divisão. Por isso usamos a **validação cruzada estratificada com 5 partições**: o treino é dividido em 5 blocos, e cada bloco serve, uma vez, como validação enquanto os outros 4 treinam o modelo. A média das 5 rodadas é a estimativa de desempenho.

A métrica principal é a **ROC AUC**. Ela mede a capacidade do modelo de ordenar os solicitantes do mais arriscado ao menos arriscado, independentemente do limiar de decisão. Como o problema é desbalanceado, a acurácia não é a melhor referência.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
SCORING = {"roc_auc": "roc_auc", "accuracy": "accuracy", "precision": "precision",
           "recall": "recall", "f1": "f1"}

## 8. Modelagem com configurações padrão

Comparamos quatro algoritmos clássicos, além de uma regressão logística como referência:

- **KNN:** classifica cada solicitante pela maioria dos vizinhos mais próximos.
- **Árvore de classificação:** cria regras do tipo "se conta < 0 e prazo > 24 meses, então…". Usamos `class_weight="balanced"` para que a árvore dê mais peso aos maus pagadores.
- **Naive Bayes:** estima a probabilidade de cada classe assumindo independência entre atributos. É uma hipótese forte, mas simples e rápida.
- **SVM (kernel RBF):** procura a fronteira que separa as classes com a maior margem possível. Também usa `class_weight="balanced"` e `probability=True`, que ativa a estimativa de probabilidades usada no restante do projeto.
- **Regressão logística:** modelo linear simples, já usado nas etapas anteriores do projeto. Serve de referência.

Primeiro rodamos cada modelo com os parâmetros padrão, só para ter um ponto de partida.

In [ ]:
MODELOS = {
    "KNN": KNeighborsClassifier(),
    "Árvore de classificação": DecisionTreeClassifier(class_weight="balanced", random_state=SEED),
    "Naive Bayes": GaussianNB(),
    "SVM": SVC(kernel="rbf", class_weight="balanced", probability=True, random_state=SEED),
    "Regressão logística (referência)": LogisticRegression(max_iter=1000, class_weight="balanced"),
}

linhas = []
for nome, modelo in MODELOS.items():
    resultado = cross_validate(criar_pipeline(modelo), X_train, y_train,
                               cv=cv, scoring=SCORING, n_jobs=-1)
    linha = {"modelo": nome}
    linha.update({metrica: resultado[f"test_{metrica}"].mean() for metrica in SCORING})
    linha["desvio_roc_auc"] = resultado["test_roc_auc"].std()
    linhas.append(linha)

baseline = pd.DataFrame(linhas).round(3)
print("Validação cruzada no treino, com parâmetros padrão:")
baseline

## 9. Otimização de hiperparâmetros

Cada algoritmo tem parâmetros que controlam sua complexidade, e os valores padrão raramente são os melhores. Usamos `GridSearchCV`, que testa todas as combinações de uma grade de valores, avaliando cada uma pela validação cruzada no treino. A métrica de otimização é a ROC AUC.

O conjunto de teste continua fora deste processo. Os hiperparâmetros são escolhidos apenas com os dados de treino.

As grades são pequenas de propósito. Com 800 linhas, grades grandes levariam a ajustes excessivos aos dados de validação.

- **KNN:** número de vizinhos e peso pela distância.
- **Árvore:** profundidade máxima e número mínimo de amostras por folha, que controlam o tamanho da árvore.
- **Naive Bayes:** `var_smoothing`, que suaviza as variâncias estimadas.
- **SVM:** `C`, que controla a penalização de erros, e `gamma`, que define o alcance de cada exemplo na fronteira.
- **Regressão logística:** `C`, a força da regularização.

In [ ]:
GRADES = {
    "KNN": {"clf__n_neighbors": [3, 5, 7, 9, 11, 15, 21],
            "clf__weights": ["uniform", "distance"]},
    "Árvore de classificação": {"clf__max_depth": [2, 3, 4, 5, 6, 8, None],
                                "clf__min_samples_leaf": [5, 10, 20, 40]},
    "Naive Bayes": {"clf__var_smoothing": [1e-9, 1e-7, 1e-5, 1e-3, 1e-1]},
    "SVM": {"clf__C": [0.1, 1, 10],
            "clf__gamma": ["scale", 0.01, 0.1]},
    "Regressão logística (referência)": {"clf__C": [0.01, 0.1, 1, 10]},
}

resultados = {}
for nome, modelo in MODELOS.items():
    busca = GridSearchCV(criar_pipeline(modelo), GRADES[nome], scoring="roc_auc",
                         cv=cv, n_jobs=-1, refit=True)
    busca.fit(X_train, y_train)
    resultados[nome] = busca
    print(f"{nome}: melhor ROC AUC na validação = {busca.best_score_:.3f} | {busca.best_params_}")

## 10. Avaliação no conjunto de teste

Agora cada modelo, já com os melhores hiperparâmetros e treinado com todo o conjunto de treino, é avaliado nos 200 casos de teste que nunca foram usados antes.

Além da ROC AUC, olhamos métricas da classe de maus pagadores:

- **Recall:** dos maus pagadores reais, quantos o modelo identificou. Em crédito, deixar passar um mau pagador costuma ser o erro mais caro.
- **Precisão:** dos solicitantes marcados como maus, quantos realmente são.
- **F1:** média harmônica entre precisão e recall.

As previsões de classe usam o limiar de 0,5 sobre a probabilidade. Esse limiar é uma escolha de convenção, e voltaremos a ele nos pontos de atenção.

In [ ]:
linhas = []
for nome, busca in resultados.items():
    modelo_ajustado = busca.best_estimator_
    probabilidade = modelo_ajustado.predict_proba(X_test)[:, 1]
    previsao = (probabilidade >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, previsao).ravel()
    linhas.append({
        "modelo": nome,
        "roc_auc": roc_auc_score(y_test, probabilidade),
        "acuracia": accuracy_score(y_test, previsao),
        "precisao_mau": precision_score(y_test, previsao),
        "recall_mau": recall_score(y_test, previsao),
        "f1_mau": f1_score(y_test, previsao),
        "verdadeiros_maus": tp,
        "maus_nao_detectados": fn,
        "bons_barrados": fp,
        "bons_aprovados": tn,
    })

avaliacao = pd.DataFrame(linhas).sort_values("roc_auc", ascending=False).round(3)
avaliacao

## 11. Comparação visual dos resultados

Os gráficos mostram duas coisas. As curvas ROC mostram o trade-off entre detectar maus pagadores e alarmes falsos, para todos os limiares possíveis. O gráfico de barras compara a estimativa obtida na validação cruzada com o resultado no teste.

Uma diferença grande entre as duas barras indicaria um modelo que se ajustou demais ao treino. Diferenças pequenas indicam que a estimativa da validação cruzada é confiável.

In [ ]:
fig, eixo = plt.subplots(figsize=(7, 6))
for nome, busca in resultados.items():
    probabilidade = busca.best_estimator_.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probabilidade)
    eixo.plot(fpr, tpr, label=f"{nome} (AUC = {roc_auc_score(y_test, probabilidade):.3f})")
eixo.plot([0, 1], [0, 1], "k--", label="Aleatório")
eixo.set_xlabel("Taxa de falsos positivos")
eixo.set_ylabel("Taxa de verdadeiros positivos (recall)")
eixo.set_title("Curvas ROC no conjunto de teste")
eixo.legend(loc="lower right", fontsize=8)
plt.show()

comparacao = pd.DataFrame({
    "Validação cruzada (treino)": {n: b.best_score_ for n, b in resultados.items()},
    "Teste (holdout)": {n: roc_auc_score(y_test, b.best_estimator_.predict_proba(X_test)[:, 1])
                        for n, b in resultados.items()},
}).sort_values("Validação cruzada (treino)")

comparacao.plot.barh(figsize=(8, 5), xlim=(0.5, 0.9), title="ROC AUC: validação cruzada x teste")
plt.xlabel("ROC AUC")
plt.tight_layout()
plt.show()

## 12. Modelo com restrições de monotonicidade

Os modelos anteriores aprendem qualquer relação entre os atributos e o risco, inclusive relações contrárias ao senso comum. Os próprios dados mostram casos assim: solicitantes **sem conta corrente** têm taxa de maus pagadores de 11,7%, menor que a dos que têm mais de 200 DM (22%).

Para tornar a lógica do modelo mais previsível, usamos um **modelo com restrições de monotonicidade**. Uma restrição obriga o risco a só subir ou só cair quando o valor de um atributo aumenta. Aplicamos essa restrição a três atributos, nos quais a direção é consistente com o senso de negócio e com os dados:

- **Tempo no emprego:** quanto mais tempo, menor o risco. Estar desempregado é o pior nível.
- **Prazo do empréstimo:** quanto maior o prazo, maior o risco.
- **Valor do crédito:** quanto maior o valor, maior o risco.

Não impomos restrição a conta corrente, histórico de crédito, finalidade, poupança e idade. Testamos restringir também a conta e o histórico na direção intuitiva, e o resultado foi ruim: a ROC AUC caiu de cerca de 0,77 para cerca de 0,70, e a probabilidade deixou de variar com essas variáveis. Ou seja, a restrição obrigou o modelo a ignorar informações que os dados contêm. Por isso, essas duas variáveis são aprendidas livremente, e os resultados contraintuitivos aparecem como achados a serem explicados.

Os atributos categóricos entram como categorias, sem ordem imposta, porque a ordem entre elas não é garantida pelos dados. A única variável ordenada é o tempo no emprego, codificada do maior para o menor tempo, com o desempregado por último.

In [ ]:
ORDEM_EMPREGO = ["A75", "A74", "A73", "A72", "A71"]  # de menor para maior risco

def criar_preprocessamento_monotono():
    return ColumnTransformer([
        ("categoricas", OrdinalEncoder(), ["status_checking_account", "credit_history",
                                           "purpose", "savings_account"]),
        ("emprego", OrdinalEncoder(categories=[ORDEM_EMPREGO]), ["employment_since"]),
        ("numericas", "passthrough", ["duration_months", "credit_amount", "age"]),
    ])

# Colunas após o pré-processamento: 0-3 categóricas, 4 emprego, 5 prazo, 6 valor, 7 idade
RESTRICOES = [0, 0, 0, 0, 1, 1, 1, 0]   # +1: risco sobe com o valor; 0: sem restrição

def criar_modelo_monotono():
    return Pipeline([
        ("prep", criar_preprocessamento_monotono()),
        ("clf", HistGradientBoostingClassifier(
            categorical_features=[0, 1, 2, 3], monotonic_cst=RESTRICOES,
            class_weight="balanced", random_state=SEED)),
    ])

GRADE_MONOTONA = {"clf__learning_rate": [0.05, 0.1], "clf__max_depth": [2, 3, None],
                  "clf__min_samples_leaf": [10, 20, 40]}

busca_monotona = GridSearchCV(criar_modelo_monotono(), GRADE_MONOTONA, scoring="roc_auc",
                              cv=cv, n_jobs=-1, refit=True)
busca_monotona.fit(X_train, y_train)
print("Modelo monotônico | melhor ROC AUC na validação =", round(busca_monotona.best_score_, 3),
      "|", busca_monotona.best_params_)
print("ROC AUC no teste:", round(roc_auc_score(y_test, busca_monotona.best_estimator_.predict_proba(X_test)[:, 1]), 3))

### Verificação da lógica do modelo

Para conferir se as restrições funcionam, variamos um atributo por vez, mantendo o mesmo perfil de solicitante. Em cada linha, o risco deve subir ou cair na direção esperada. Nos atributos sem restrição, a variação pode ter qualquer direção, e isso é esperado.

In [ ]:
perfil = dict(status_checking_account="A11", credit_history="A32", purpose="A43",
              savings_account="A61", employment_since="A73", duration_months=24,
              credit_amount=5000.0, age=35)
modelo_mono = busca_monotona.best_estimator_

def probabilidade(**mudanca):
    registro = pd.DataFrame([{**perfil, **mudanca}])
    return round(float(modelo_mono.predict_proba(registro)[0, 1]), 3)

print("Emprego (restrito):", {e: probabilidade(employment_since=e) for e in ORDEM_EMPREGO})
print("Prazo (restrito):  ", {d: probabilidade(duration_months=d) for d in [6, 12, 24, 48]})
print("Valor (restrito):  ", {v: probabilidade(credit_amount=v) for v in [1000.0, 5000.0, 15000.0]})
print("Conta (livre):     ", {c: probabilidade(status_checking_account=c) for c in ["A11", "A12", "A13", "A14"]})
print("Histórico (livre): ", {h: probabilidade(credit_history=h) for h in ["A30", "A31", "A32", "A33", "A34"]})

## 13. Escolha e exportação do modelo

O modelo exportado é o **modelo com restrições de monotonicidade**. Essa escolha veio de uma exigência do projeto, e não só da métrica: queremos um modelo cuja lógica seja previsível nas variáveis de emprego, prazo e valor. Por isso, a escolha não segue apenas o maior ROC AUC.

A comparação com os outros algoritmos continua registrada na tabela da seção 11. Quando o modelo monotônico fica próximo do melhor modelo sem restrição, a restrição tem custo pequeno. Esse custo é medido na validação cruzada, e não no teste, para não usar o conjunto de teste para decidir.

Depois de escolhido, o pipeline completo, com codificação e modelo, é salvo em um único arquivo. Isso garante que qualquer aplicação que carregue o arquivo aplique exatamente as mesmas transformações do treino.

In [ ]:
modelo_final = busca_monotona.best_estimator_
melhor_sem_restricao = max(["KNN", "Árvore de classificação", "Naive Bayes", "SVM"],
                           key=lambda nome: resultados[nome].best_score_)

print("Modelo exportado: HistGradientBoosting com restrições de monotonicidade")
print("Hiperparâmetros:", busca_monotona.best_params_)
print(f"ROC AUC na validação cruzada: monotônico {busca_monotona.best_score_:.3f} | "
      f"melhor sem restrição ({melhor_sem_restricao}) {resultados[melhor_sem_restricao].best_score_:.3f}")

ARQUIVO = "modelo_risco_credito.joblib"
joblib.dump(modelo_final, ARQUIVO)
print("Modelo salvo em:", ARQUIVO)

try:
    from google.colab import files
    files.download(ARQUIVO)
except ImportError:
    print("Fora do Colab, o arquivo fica na pasta de execução.")

## 14. Teste do modelo exportado

Carregamos o arquivo salvo e comparamos as probabilidades com as do modelo original. Se os valores forem idênticos, a exportação preservou o modelo. Também conferimos os nomes das etapas do pipeline, porque a API do projeto acessa a etapa de pré-processamento pelo nome `prep`.

In [ ]:
modelo_carregado = joblib.load(ARQUIVO)
amostra = X_test.iloc[:5]

p_original = modelo_final.predict_proba(amostra)[:, 1]
p_carregado = modelo_carregado.predict_proba(amostra)[:, 1]

print("Probabilidades (original): ", np.round(p_original, 4))
print("Probabilidades (carregado):", np.round(p_carregado, 4))
print("Idênticas:", np.allclose(p_original, p_carregado))
print("Etapas do pipeline:", list(modelo_carregado.named_steps))
print("Versão do scikit-learn usada:", sklearn.__version__)

## 15. Principais achados e pontos de atenção

**Principais achados**

- **Os algoritmos têm desempenho próximo.** Na validação cruzada, SVM (0,792), modelo monotônico (0,788), Naive Bayes (0,772), regressão logística (0,770) e KNN (0,766) ficam em uma faixa estreita. A árvore de decisão é a mais fraca (0,729), o que sugere que regras rígidas captam menos desse problema.
- **A restrição de monotonicidade custa pouco na validação cruzada.** O modelo monotônico fica 0,004 abaixo do SVM. No teste, a diferença é maior: 0,783 contra 0,797 do SVM e 0,817 da regressão logística. Com apenas 60 maus pagadores no teste, essas diferenças não são conclusivas, mas a regressão logística é a melhor no teste. Essa diferença não muda a escolha, que segue a exigência de lógica previsível e a validação cruzada.
- **Restringir conta e histórico tem custo alto.** Em testes exploratórios, impor a direção intuitiva nessas duas variáveis derrubou a ROC AUC para cerca de 0,70. Por isso, a restrição ficou só em emprego, prazo e valor.
- **Os resultados contraintuitivos vêm dos dados.** Solicitantes sem conta corrente têm 11,7% de maus pagadores, contra 22% dos que têm mais de 200 DM. Clientes com conta crítica (A34) têm 17%, o menor índice da variável de histórico. O modelo reproduz esses padrões, e eles não devem ser lidos como causa. Uma explicação possível é o viés de seleção: a base contém apenas empréstimos aprovados, e a decisão de conceder crédito já filtrou parte do risco.

**Pontos de atenção**

- **O limiar de 0,5 precisa ser definido pelo negócio.** Com esse limiar, o modelo monotônico identifica 68% dos maus pagadores no teste (recall), com precisão de 50%. Um banco que prefira barrar menos bons pagadores pode ajustar o limiar, e isso é uma decisão de custo, não de estatística.
- **O recall de maus pagadores varia muito entre os modelos.** A regressão logística identifica 78% dos maus pagadores, contra 45% do SVM e 38% do KNN. Modelos com recall alto barram mais bons pagadores.
- **Os atributos sensíveis foram excluídos.** `personal_status_sex` e `foreign_worker` não entram no modelo. Essa escolha pode reduzir um pouco o desempenho, mas evita que o modelo aprenda discriminação por sexo ou nacionalidade.
- **A base é antiga e pequena.** Os dados são de 1994, com 1000 registros e valores em marco alemão. Um modelo para uso real precisaria de dados atuais e de validação em produção.

Os números deste notebook são de uma execução de referência. Pequenas diferenças podem aparecer entre versões do scikit-learn, mas a ordem geral dos resultados deve se manter.

## 16. Conclusão

Este notebook percorreu o ciclo completo de um problema de classificação com métodos clássicos: carga, exploração, separação entre treino e teste, transformação com pipelines, validação cruzada, otimização de hiperparâmetros, avaliação no holdout, comparação e exportação.

O modelo exportado é um HistGradientBoosting com restrições de monotonicidade em emprego, prazo e valor. Ele tem desempenho próximo ao melhor modelo sem restrição na validação cruzada, e sua lógica nessas três variáveis é previsível. Os resultados contraintuitivos de conta e histórico foram mantidos, porque vêm dos dados, e não foram forçados por uma restrição.

O resultado mais importante, porém, não é o algoritmo. É a combinação de um bom ranqueamento de risco, um limiar definido pelo custo dos erros e a atenção às limitações dos dados. Esses são os próximos passos antes de qualquer uso real.